ТЕСТ

(Мы добавили openpyxl для чтения Excel и pyarrow для сохранения в parquet, которые нужны нашему пайплайну).

uv add pandas pymorphy2 nltk scikit-learn openpyxl pyarrow
uv add ipykernel

uv run python -m ipykernel install --user --name=lab6 --display-name="Python (uv - lab6)"

$env:NLTK_ALLOW_PROXIED_URLOPEN="1"
uv run python -c "import nltk; nltk.download('stopwords')"

In [1]:
import sys
import os
import pandas as pd

# Добавляем корень проекта в sys.path, чтобы Python видел папку src
# os.path.abspath('..') поднимается из notebooks/ в корень проекта
sys.path.append(os.path.abspath('..'))

# Импортируем наш модуль
from src.preprocessing import clean_text, lemmatize_text, preprocess_dataframe

print("Импорт прошел успешно!")

Импорт прошел успешно!


In [2]:
from src.preprocessing import clean_text, fix_typos, lemmatize_text

test_text = "Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим."

print("=== ИСХОДНЫЙ ТЕКСТ ===")
print(test_text)

# 1. Очистка
cleaned = clean_text(test_text)
print("\n=== ПОСЛЕ clean_text ===")
print(cleaned)

# 2. Исправление опечаток
fixed = fix_typos(cleaned)
print("\n=== ПОСЛЕ fix_typos ===")
print(fixed)

# 3. Лемматизация
lemmatized = lemmatize_text(fixed)
print("\n=== ПОСЛЕ lemmatize_text ===")
print(lemmatized)

=== ИСХОДНЫЙ ТЕКСТ ===
Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим.

=== ПОСЛЕ clean_text ===
не хватило <num> балла по биологии может ли она пересдать экзамен по билогии

=== ПОСЛЕ fix_typos ===
не хватило <num> балла по биологии может ли она пересдать экзамен по биология

=== ПОСЛЕ lemmatize_text ===
хватить <num> балл биология пересдать экзамен биология


In [3]:
# 1. Загружаем данные (путь от папки notebooks/)
df = pd.read_excel('../data/mails.xlsx')
print(f"Загружено строк: {df.shape[0]}")

# 2. Запускаем пайплайн
df_processed = preprocess_dataframe(df)

# 3. Смотрим, что получилось (первые 3 строки)
display(df_processed[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm']].head(3))

Загружено строк: 588
1. Применяем базовую очистку...
2. Исправляем опечатки...
3. Применяем лемматизацию (может занять пару минут)...
4. Применяем замену сленга...


,CONTENT,CONTENT_clean,CONTENT_lemm
0,Здравствуйте! Поступила в ваш институт на бюдж...,поступила в ваш институт на бюджет педиатричес...,поступить ваш институт бюджет педиатрический ф...
1,Здравствуйте! Я прошла конкурс по трем дисципл...,я прошла конкурс по трем дисциплинам зачислена...,пройти конкурс три дисциплина зачислить дисцип...
2,Здравствуйте! Скажите пожалуйста могу ли я под...,скажите пожалуйста могу ли я подать документы ...,сказать пожалуйста мочь подать документ направ...


In [4]:
import pandas as pd

# Отключаем обрезку текста в колонках
pd.set_option('display.max_colwidth', None)

# Твой код вывода
display(df_processed[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm']].head(3))

,CONTENT,CONTENT_clean,CONTENT_lemm
0,"Здравствуйте! Поступила в ваш институт на бюджет Педиатрического факультета. Написала отказ от зачисления, как удостоверится в том, что меня отчислили? С уважнием, Ксения.",поступила в ваш институт на бюджет педиатрического факультета написала отказ от зачисления как удостоверится в том что меня отчислили,поступить ваш институт бюджет педиатрический факультет написать отказ зачисление удостовериться отчислить
1,"Здравствуйте! Я прошла конкурс по трем дисциплинам, зачислена дисциплину по #-й приоритету, возможно ли перейти сейчас на другой приоритет(управление персоналом) или ""поезд ушел""?",я прошла конкурс по трем дисциплинам зачислена дисциплину по <num> й приоритету возможно ли перейти сейчас на другой приоритет управление персоналом или поезд ушел,пройти конкурс три дисциплина зачислить дисциплина <num> приоритет возможно перейти приоритет управление персонал поезд уйти
2,"Здравствуйте! Скажите пожалуйста могу ли я подать документы на направление Экология если в этом году ЕГЭ в ## классе Не сдавала есть результаты года за # класс русского языка, математики, и география.",скажите пожалуйста могу ли я подать документы на направление экология если в этом году егэ в <num> классе не сдавала есть результаты года за <num> класс русского языка математики и география,сказать пожалуйста мочь подать документ направление экология год егэ <num> класс сдавать результат год <num> класс русский язык математика география


In [5]:
from src.preprocessing import split_and_save_data

train_df, test_df = split_and_save_data(df_processed)


5. Делим данные на Train/Test (0.9/0.1)...
Train сохранен: 529 строк
Test сохранен:  59 строк
Файлы train.parquet и test.parquet готовы к использованию коллегами!


Как понять, что всё сработало

1. В терминале не должно быть красных ошибок (Traceback).
2. В папке data/ появятся два новых файла: train.parquet и test.parquet.
3. В ноутбуке вы увидите примеры «До/После» обработки текста.
4. Если открыть train.parquet (например, через pd.read_parquet('../data/train.parquet')), там будут колонки CONTENT_clean, CONTENT_lemm и CONTENT_edited.